In [ ]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
import lyricsgenius

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "app").exists():
            return candidate
    raise FileNotFoundError("Could not locate the BTS Song Atlas project root.")


PROJECT_ROOT = find_project_root()


In [ ]:
BASE_DIR = PROJECT_ROOT
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_DIR = BASE_DIR / "data" / "raw" / "core"

tracks = pd.read_csv(PROCESSED_DIR / "spotify_tracks_clean.csv")

tracks.head()


In [ ]:
load_dotenv(BASE_DIR / ".env")

GENIUS_ACCESS_TOKEN = os.getenv("GENIUS_ACCESS_TOKEN")

if not GENIUS_ACCESS_TOKEN:
    raise ValueError("GENIUS_ACCESS_TOKEN is missing from .env")

genius = lyricsgenius.Genius(
    GENIUS_ACCESS_TOKEN,
    timeout=15,
    retries=3,
    remove_section_headers=True
)

genius.verbose = False

In [ ]:
song = genius.search_song("Idol", "BTS")

print(song.title)
print(song.artist)
print(song.url)
print(song.lyrics[:1000])

In [ ]:
test_songs = ["ON", "Dynamite", "Black Swan", "Blood Sweat & Tears"]

for title in test_songs:
    song = genius.search_song(title, "BTS")

    print("\n---")
    print("Search:", title)

    if song:
        print("Found:", song.title)
        print("Artist:", song.artist)
        print("URL:", song.url)
        print(song.lyrics[:200])
    else: 
        print("Not found")

In [ ]:
from difflib import SequenceMatcher
import re
import unicodedata

def normalize_title(title):
    if pd.isna(title):
        return ""

    title = unicodedata.normalize("NFKC", str(title)).lower()
    title = title.replace("&", " and ")
    title = re.sub(r"[‘’´`]", "'", title)
    title = title.replace("[", "(").replace("]", ")")
    title = title.replace("{", "(").replace("}", ")")
    title = re.sub(r"[–—−]", "-", title)
    title = re.sub(r"[^0-9a-z가-힣ぁ-ゔァ-ヴー一-龯()\s/&-]", " ", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title

def title_variants(title):
    normalized = normalize_title(title)

    if not normalized:
        return set()

    variants = {normalized}

    without_parentheses = re.sub(r"\([^)]*\)", " ", normalized)
    without_parentheses = re.sub(r"\s+", " ", without_parentheses).strip(" -")
    if without_parentheses:
        variants.add(without_parentheses)

    for part in re.findall(r"\(([^)]*)\)", normalized):
        part = re.sub(r"\s+", " ", part).strip(" -")
        if part:
            variants.add(part)
            for slash_part in part.split("/"):
                slash_part = slash_part.strip()
                if slash_part:
                    variants.add(slash_part)

    expanded = set()
    for variant in variants:
        simplified = variant
        simplified = re.sub(r"\bjapanese version\b", "japanese ver", simplified)
        simplified = re.sub(r"\bjapanese ver\.\b", "japanese ver", simplified)
        simplified = re.sub(r"\bversion\b", "ver", simplified)
        simplified = re.sub(r"\bver\.\b", "ver", simplified)
        simplified = re.sub(r"\bfull length edition\b", "full length", simplified)
        simplified = re.sub(r"\s+", " ", simplified).strip(" -")
        if simplified:
            expanded.add(simplified)
        for hyphen_part in re.split(r"\s+-\s+", simplified):
            hyphen_part = hyphen_part.strip()
            if hyphen_part:
                expanded.add(hyphen_part)

    return expanded

def similarity(a, b):
    variants_a = title_variants(a)
    variants_b = title_variants(b)

    if not variants_a or not variants_b:
        return 0

    return max(
        SequenceMatcher(None, left, right).ratio()
        for left in variants_a
        for right in variants_b
    )

RAW_VALID_ARTISTS = {
    "BTS",
    "RM",
    "RM (알엠)",
    "Jin",
    "Jin (진)",
    "SUGA",
    "SUGA (슈가)",
    "Agust D",
    "j-hope",
    "j-hope (제이홉)",
    "jhope",
    "Jimin",
    "Jimin (지민)",
    "V",
    "V (뷔)",
    "Taehyung",
    "Jung Kook",
    "Jung Kook (정국)",
    "Jungkook",
    "Jungkook (정국)"
}

RAW_TRANSLATION_ARTISTS = {
    "Genius Romanizations",
    "Genius English Translations"
}

def normalize_artist_name(artist_name):
    if pd.isna(artist_name):
        return ""

    artist_name = unicodedata.normalize("NFKC", str(artist_name)).lower()
    artist_name = artist_name.replace("&", " and ")
    artist_name = re.sub(r"[^0-9a-z가-힣ぁ-ゔァ-ヴー一-龯\s]", " ", artist_name)
    artist_name = re.sub(r"\s+", " ", artist_name).strip()
    return artist_name

VALID_ARTISTS = {normalize_artist_name(name) for name in RAW_VALID_ARTISTS}
TRANSLATION_ARTISTS = {normalize_artist_name(name) for name in RAW_TRANSLATION_ARTISTS}

def classify_artist(artist_name):
    normalized_artist = normalize_artist_name(artist_name)

    if normalized_artist in VALID_ARTISTS:
        return "valid"

    if normalized_artist in TRANSLATION_ARTISTS:
        return "translation"

    return "invalid"

In [ ]:
test_songs = ["ON", "Dynamite", "Black Swan", "Blood Sweat & Tears"]

for title in test_songs:
    song = genius.search_song(title, "BTS")

    print("\n---")
    print("Search:", title)

    if song:
        title_score = similarity(title, song.title)
        artist_class = classify_artist(song.artist)
        artist_ok = artist_class == "valid"

        print("Found:", song.title)
        print("Artist:", song.artist)
        print("Title score:", round(title_score, 2))
        print("Artist OK:", artist_ok)
        print("URL:", song.url)

        if artist_class == "valid" and title_score >= 0.75:
            print("Status: exact")
        elif artist_class == "translation":
            print("Status: manual_review")
        elif artist_class == "valid" and title_score >= 0.45:
            print("Status: manual_review")
        else:
            print("Status: rejected")
    else:
        print("Status: not_found")

In [ ]:
RAW_VALID_ARTISTS = {
    "BTS",
    "RM",
    "RM (알엠)",
    "Jin",
    "Jin (진)",
    "SUGA",
    "SUGA (슈가)",
    "Agust D",
    "j-hope",
    "j-hope (제이홉)",
    "jhope",
    "Jimin",
    "Jimin (지민)",
    "V",
    "V (뷔)",
    "Taehyung",
    "Jung Kook",
    "Jung Kook (정국)",
    "Jungkook",
    "Jungkook (정국)"
}

RAW_TRANSLATION_ARTISTS = {
    "Genius Romanizations",
    "Genius English Translations"
}

def normalize_artist_name(artist_name):
    if pd.isna(artist_name):
        return ""

    artist_name = unicodedata.normalize("NFKC", str(artist_name)).lower()
    artist_name = artist_name.replace("&", " and ")
    artist_name = re.sub(r"[^0-9a-z가-힣ぁ-ゔァ-ヴー一-龯\s]", " ", artist_name)
    artist_name = re.sub(r"\s+", " ", artist_name).strip()
    return artist_name

VALID_ARTISTS = {normalize_artist_name(name) for name in RAW_VALID_ARTISTS}
TRANSLATION_ARTISTS = {normalize_artist_name(name) for name in RAW_TRANSLATION_ARTISTS}

def classify_artist(artist_name):
    normalized_artist = normalize_artist_name(artist_name)

    if normalized_artist in VALID_ARTISTS:
        return "valid"

    if normalized_artist in TRANSLATION_ARTISTS:
        return "translation"

    return "invalid"

In [ ]:
def is_instrumental(track_name):
    return "instrumental" in str(track_name).lower()

def empty_result(status):
    return {
        "genius_title": None,
        "genius_artist": None,
        "genius_url": None,
        "lyrics": None,
        "title_score": 0,
        "artist_ok": False,
        "search_status": status
    }

def evaluate_match(track_name, song):
    title_score = similarity(track_name, song.title)
    artist_class = classify_artist(song.artist)
    artist_ok = artist_class == "valid"

    if artist_class == "valid" and title_score >= 0.75:
        status = "exact"
    elif artist_class == "translation":
        status = "manual_review"
    elif artist_class == "valid" and title_score >= 0.45:
        status = "manual_review"
    else:
        status = "rejected"

    return {
        "genius_title": song.title,
        "genius_artist": song.artist,
        "genius_url": song.url,
        "lyrics": song.lyrics,
        "title_score": round(title_score, 3),
        "artist_ok": artist_ok,
        "search_status": status
    }

def search_genius_track(track_name, artist_name="BTS"):
    if is_instrumental(track_name):
        return empty_result("instrumental")

    candidates = []
    candidate_ids = set()

    first_song = genius.search_song(track_name, artist_name)
    if first_song is not None:
        candidates.append(first_song)
        first_song_id = getattr(first_song, "id", None)
        if first_song_id is not None:
            candidate_ids.add(first_song_id)

    needs_fallback = True
    if candidates:
        first_result = evaluate_match(track_name, candidates[0])
        needs_fallback = first_result["search_status"] != "exact"

    if needs_fallback:
        search_response = genius.search_songs(f"{track_name} {artist_name}", per_page=5)
        for hit in search_response.get("hits", []):
            result = hit.get("result", {})
            song_id = result.get("id")
            if song_id is None or song_id in candidate_ids:
                continue

            candidate_song = genius.search_song(song_id=song_id)
            if candidate_song is None:
                continue

            candidates.append(candidate_song)
            candidate_ids.add(song_id)

    if not candidates:
        return empty_result("not_found")

    status_rank = {
        "exact": 3,
        "manual_review": 2,
        "rejected": 1,
        "not_found": 0,
        "instrumental": 0
    }

    scored_candidates = [evaluate_match(track_name, song) for song in candidates]
    best_match = max(
        scored_candidates,
        key=lambda item: (
            status_rank[item["search_status"]],
            item["title_score"],
            len(normalize_title(item["genius_title"]))
        )
    )

    return best_match

In [ ]:
for title in ["ON", "Dynamite", "Black Swan", "Blood Sweat & Tears"]:
    result = search_genius_track(title)

    print("\n---")
    print(title)
    print(result["genius_title"])
    print(result["genius_artist"])
    print(result["title_score"])
    print(result["artist_ok"])
    print(result["search_status"])

In [ ]:
tricky_titles = [
    "Blood Sweat & Tears",
    "Dimple",
    "The Truth Untold",
    "21st Century Girl",
    "MIC Drop (Steve Aoki Remix) (Full Length Edition)",
    "No More Dream - Japanese Ver.",
    "Hormone Sensou - Japanese Ver.",
    "Euphoria",
    "Moon",
    "Filter",
    "Intro: Boy Meets Evil"
]

sample_tracks = (
    tracks[tracks["track_name"].isin(tricky_titles)]
    .drop_duplicates(subset=["track_name"])
    .copy()
)

results = []

for _, row in sample_tracks.iterrows():
    result = search_genius_track(row["track_name"])

    results.append({
        "track_id": row["track_id"],
        "track_name": row["track_name"],
        "album_id": row["album_id"],
        "album_name": row["album_name"],
        **result
    })

lyrics_sample = pd.DataFrame(results)

lyrics_sample[[
    "track_name",
    "genius_title",
    "genius_artist",
    "title_score",
    "artist_ok",
    "search_status"
]].sort_values("track_name").reset_index(drop=True)

In [ ]:
lyrics_sample["search_status"].value_counts()

In [ ]:
lyrics_sample[
    lyrics_sample["search_status"] != "exact"
][["track_name", "genius_title", "genius_artist", "title_score", "artist_ok", "search_status"]]

In [ ]:
sample_tracks = tracks.sample(30, random_state=42).copy()

In [ ]:
is_instrumental("SWIM (Instrumental)"), is_instrumental("Dynamite")

In [ ]:
sample_tracks = tracks.sample(30, random_state=42).copy()

results = []

for _, row in sample_tracks.iterrows():
    result = search_genius_track(row["track_name"])

    results.append({
        "track_id": row["track_id"],
        "track_name": row["track_name"],
        "album_id": row["album_id"],
        "album_name": row["album_name"],
        **result
    })

lyrics_sample = pd.DataFrame(results)

lyrics_sample[[
    "track_name",
    "genius_title",
    "genius_artist",
    "title_score",
    "artist_ok",
    "search_status"
]]

In [ ]:
lyrics_sample["search_status"].value_counts()

In [ ]:
lyrics_sample[
    lyrics_sample["search_status"] != "exact"
][["track_name", "genius_title", "genius_artist", "title_score", "artist_ok", "search_status"]]

In [ ]:
sample_tracks_100 = tracks.sample(100, random_state=123).copy()

results = []

for _, row in sample_tracks_100.iterrows():
    result = search_genius_track(
        row["track_name"],
        "BTS"
    )

    results.append({
        "track_id": row["track_id"],
        "track_name": row["track_name"],
        "album_id": row["album_id"],
        "album_name": row["album_name"],
        **result
    })

lyrics_sample_100 = pd.DataFrame(results)

In [ ]:
lyrics_sample_100["search_status"].value_counts()

In [ ]:
lyrics_sample_100[
    lyrics_sample_100["search_status"] != "exact"
][[
    "track_name",
    "genius_title",
    "genius_artist",
    "title_score",
    "search_status"
]]

In [ ]:
status_counts = lyrics_sample_100["search_status"].value_counts(normalize=True) * 100
print(status_counts.round(1))

In [ ]:
from time import sleep
from tqdm.notebook import tqdm

RAW_DIR.mkdir(parents=True, exist_ok=True)

all_results = []

for _, row in tqdm(tracks.iterrows(), total=len(tracks)):
    result = search_genius_track(row["track_name"], "BTS")

    all_results.append({
        "track_id": row["track_id"],
        "track_name": row["track_name"],
        "album_id": row["album_id"],
        "album_name": row["album_name"],
        "genius_title": result["genius_title"],
        "genius_artist": result["genius_artist"],
        "genius_url": result["genius_url"],
        "lyrics": result["lyrics"],
        "title_score": result["title_score"],
        "artist_ok": result["artist_ok"],
        "search_status": result["search_status"]
    })

    sleep(0.5)

In [ ]:
genius_lyrics_raw = pd.DataFrame(all_results)

genius_lyrics_raw.to_csv(
    RAW_DIR / "genius_lyrics_raw.csv",
    index=False
)

print("Saved:", RAW_DIR / "genius_lyrics_raw.csv")
print("Shape:", genius_lyrics_raw.shape)

In [ ]:
genius_lyrics_raw["search_status"].value_counts()

In [ ]:
genius_lyrics_raw[
    genius_lyrics_raw["search_status"] != "exact"
][[
    "track_name",
    "genius_title",
    "genius_artist",
    "title_score",
    "artist_ok",
    "search_status"
]]

In [ ]:
check = pd.read_csv(RAW_DIR / "genius_lyrics_raw.csv")

print(check.shape)
check["search_status"].value_counts()